In [4]:
import os

DATA_DIR = "data"

print("Searching for files under:", os.path.abspath(DATA_DIR))
print("=" * 70)

for root, dirs, files in os.walk(DATA_DIR):
    for f in files:
        full_path = os.path.join(root, f)
        size_kb = os.path.getsize(full_path) / 1024
        print(f"{full_path}   ({size_kb:.1f} KB)")

Searching for files under: /home/jovyan/QS-Agric-T3/data
data/Agr data/NAROK DAILY MAX AND MIN TEMPERATURE.xls   (175.0 KB)
data/Agr data/BOMET potential water deficit.xlsx   (65.5 KB)
data/Agr data/UASIN GISHU SOIL MOISTURE.xlsx 2016-2020.xlsx   (10.7 KB)
data/Agr data/NAKURU MEAN DAILY RAINFALL.xls 2016-2020.xls   (223.5 KB)
data/Agr data/BUNGOMA SOIL MOISTURE.xlsx 2016-2020.xlsx   (10.7 KB)
data/Agr data/TRANSNZOIA potential water deficit.xlsx   (65.7 KB)
data/Agr data/NAKURU potential water deficit.xlsx   (66.1 KB)
data/Agr data/NANDI DAILY MAX AND MIN TEMPERATURE.xls   (174.5 KB)
data/Agr data/BUNGOMA DAILY MAX AND MIN DAILY TEMPERATURE.xls   (175.0 KB)
data/Agr data/NANDI MEAN DAILY RAINFALL.xlsx 2016-2020.xlsx   (63.7 KB)
data/Agr data/Annual Maize Yield Production 2012-2020.xlsx   (53.9 KB)
data/Agr data/MARAKWET DAILY MAX AND MIN TEMPERATURE.xls   (174.5 KB)
data/Agr data/BUNGOMA MEAN DAILY RAINFALL.xlsx 2016-2020.xls   (274.0 KB)
data/Agr data/TRANSNZOIA DAILY MAX AND MIN TEM

In [3]:
import pandas as pd

yield_path = "data/Agr data/Annual Maize Yield Production 2012-2020.xlsx"

def parse_sheet1(path):
    raw = pd.read_excel(path, sheet_name="Sheet1", header=None)
    year_row = raw.iloc[1]
    data = raw.iloc[3:].reset_index(drop=True)

    records = []
    n_cols = raw.shape[1]
    col = 1
    while col < n_cols:
        year = year_row[col]
        if pd.isna(year):
            col += 1
            continue
        harvested_col, production_col, yield_col = col, col + 1, col + 2
        for _, row in data.iterrows():
            county = row[0]
            if pd.isna(county):
                continue
            records.append({
                "county": str(county).strip(),
                "year": int(year),
                "harvested_area_ha": row[harvested_col],
                "production_tonnes": row[production_col],
                "yield_t_ha": row[yield_col],
            })
        col += 3
    return pd.DataFrame(records)


def parse_sheet4(path):
    raw = pd.read_excel(path, sheet_name="Sheet4")
    raw.columns = ["county", "year", "indicator", "value"]
    pivoted = raw.pivot_table(
        index=["county", "year"], columns="indicator", values="value", aggfunc="first"
    ).reset_index()
    pivoted.columns.name = None
    pivoted = pivoted.rename(columns={
        "Area (HA)": "harvested_area_ha",
        "Production (MT)": "production_tonnes",
        "Yield(MT/HA)": "yield_t_ha",
    })
    pivoted["county"] = pivoted["county"].str.strip()
    return pivoted


sheet1_df = parse_sheet1(yield_path)
sheet4_df = parse_sheet4(yield_path)

print("Sheet1 years:", sorted(sheet1_df["year"].unique()))
print("Sheet1 counties:", sorted(sheet1_df["county"].unique()))
print("\nSheet4 years:", sorted(sheet4_df["year"].unique()))
print("Sheet4 counties:", sorted(sheet4_df["county"].unique()))

combined = pd.concat([sheet4_df, sheet1_df], ignore_index=True)
combined = combined.drop_duplicates(subset=["county", "year"], keep="first")
combined = combined.sort_values(["county", "year"]).reset_index(drop=True)

print("\nCombined shape:", combined.shape)
print("Combined years:", sorted(combined["year"].unique()))
print("Combined counties:", sorted(combined["county"].unique()))
print("\nSample:")
print(combined.head(10))

Sheet1 years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2020)]
Sheet1 counties: ['Bomet', 'Bungoma', 'Elgeyo/Marakwet', 'Kakamega', 'Nakuru', 'Nandi', 'Narok', 'Trans Nzoia', 'Uasin Gishu']

Sheet4 years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2020)]
Sheet4 counties: ['Baringo', 'Bomet', 'Bungoma', 'Busia', 'Elgeyo/Marakwet', 'Embu', 'Garissa', 'Homabay', 'Isiolo', 'Kajiado', 'Kakamega', 'Kericho', 'Kiambu', 'Kilifi', 'Kirinyaga', 'Kisii', 'Kisumu', 'Kitui', 'Kwale', 'Laikipia', 'Lamu', 'Machakos', 'Makueni', 'Mandera', 'Marsabit', 'Meru', 'Migori', 'Mombasa', "Murang'a", 'Nairobi', 'Nakuru', 'Nandi', 'Narok', 'Nyamira', 'Nyandarua', 'Nyeri', 'Samburu', 'Siaya', 'Taita/Taveta', 'Tana River', 'Tharaka-Nthi', 'Trans Nzoia', 'Turkana', 'Uasin Gishu', 'Vihiga', 'Wajir', 'West Pokot']

Combined shape: (376, 5)
Combined years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.

In [2]:
!pip install pandas numpy matplotlib scikit-learn qiskit qiskit-machine-learning qiskit-algorithms openpyxl xlrd

  Using cached pandas-3.0.5-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached matplotlib-3.11.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached scikit_learn-1.9.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached qiskit_machine_learning-0.9.1-py3-none-any.whl.metadata (13 kB)
  Using cached qiskit_algorithms-0.4.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached xlrd-2.0.2-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.64.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (123 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.2 k

In [5]:
import pandas as pd

def parse_yield_sheet(path, sheet_name):
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)

    year_row = raw.iloc[0]
    data = raw.iloc[2:].reset_index(drop=True)

    records = []
    n_cols = raw.shape[1]
    col = 1
    while col < n_cols:
        year = year_row[col]
        if pd.isna(year):
            col += 1
            continue
        harvested_col, production_col, yield_col = col, col + 1, col + 2

        for _, row in data.iterrows():
            county = row[0]
            if pd.isna(county):
                continue
            records.append({
                "county": str(county).strip(),
                "year": int(year),
                "harvested_area_ha": row[harvested_col],
                "production_tonnes": row[production_col],
                "yield_t_ha": row[yield_col],
            })
        col += 3

    return pd.DataFrame(records)


yield_path = "data/Agr data/Annual Maize Yield Production 2012-2020.xlsx"
sheet1_df = parse_yield_sheet(yield_path, "Sheet1")
sheet4_df = parse_yield_sheet(yield_path, "Sheet4")

combined = pd.concat([sheet1_df, sheet4_df], ignore_index=True)
combined = combined.drop_duplicates(subset=["county", "year"])
combined = combined.sort_values(["county", "year"]).reset_index(drop=True)

print("Sheet1 years found:", sorted(sheet1_df["year"].unique()))
print("Sheet4 years found:", sorted(sheet4_df["year"].unique()))
print("\nCombined shape:", combined.shape)
print("\nCombined sample:")
print(combined.head(15))
print("\nAll years covered:", sorted(combined["year"].unique()))
print("All counties covered:", sorted(combined["county"].unique()))

ValueError: invalid literal for int() with base 10: 'Year'

In [6]:
raw4 = pd.read_excel(yield_path, sheet_name="Sheet4", header=None)
print("Shape:", raw4.shape)
print(raw4.iloc[:5]) 

Shape: (1129, 4)
         0     1                2         3
0   County  Year        Indicator     Value
1  Baringo  2012        Area (HA)     39753
2  Baringo  2012  Production (MT)  71866.62
3  Baringo  2012     Yield(MT/HA)  1.807829
4  Baringo  2013        Area (HA)     29117


In [7]:
import pandas as pd

yield_path = "data/Agr data/Annual Maize Yield Production 2012-2020.xlsx"

# ---------------------------------------------------------
# Sheet1: wide, multi-year-block format (already working)
# ---------------------------------------------------------
def parse_sheet1(path):
    raw = pd.read_excel(path, sheet_name="Sheet1", header=None)
    year_row = raw.iloc[0]
    data = raw.iloc[2:].reset_index(drop=True)

    records = []
    n_cols = raw.shape[1]
    col = 1
    while col < n_cols:
        year = year_row[col]
        if pd.isna(year):
            col += 1
            continue
        harvested_col, production_col, yield_col = col, col + 1, col + 2
        for _, row in data.iterrows():
            county = row[0]
            if pd.isna(county):
                continue
            records.append({
                "county": str(county).strip(),
                "year": int(year),
                "harvested_area_ha": row[harvested_col],
                "production_tonnes": row[production_col],
                "yield_t_ha": row[yield_col],
            })
        col += 3
    return pd.DataFrame(records)


# ---------------------------------------------------------
# Sheet4: long format, needs pivoting
# County | Year | Indicator | Value  ->  one row per county-year
# ---------------------------------------------------------
def parse_sheet4(path):
    raw = pd.read_excel(path, sheet_name="Sheet4")  # headers already correct
    raw.columns = ["county", "year", "indicator", "value"]

    pivoted = raw.pivot_table(
        index=["county", "year"],
        columns="indicator",
        values="value",
        aggfunc="first"
    ).reset_index()

    pivoted.columns.name = None
    pivoted = pivoted.rename(columns={
        "Area (HA)": "harvested_area_ha",
        "Production (MT)": "production_tonnes",
        "Yield(MT/HA)": "yield_t_ha",
    })
    pivoted["county"] = pivoted["county"].str.strip()
    return pivoted


sheet1_df = parse_sheet1(yield_path)
sheet4_df = parse_sheet4(yield_path)

print("Sheet1 years:", sorted(sheet1_df["year"].unique()))
print("Sheet1 counties:", sorted(sheet1_df["county"].unique()))
print("\nSheet4 years:", sorted(sheet4_df["year"].unique()))
print("Sheet4 counties:", sorted(sheet4_df["county"].unique()))

# Combine, preferring Sheet4 where both overlap (Sheet4 looked cleaner),
# but keep whichever years each sheet uniquely provides
combined = pd.concat([sheet4_df, sheet1_df], ignore_index=True)
combined = combined.drop_duplicates(subset=["county", "year"], keep="first")
combined = combined.sort_values(["county", "year"]).reset_index(drop=True)

print("\nCombined shape:", combined.shape)
print("Combined years:", sorted(combined["year"].unique()))
print("Combined counties:", sorted(combined["county"].unique()))
print("\nSample:")
print(combined.head(10))

KeyError: 'year'

In [8]:
raw1 = pd.read_excel(yield_path, sheet_name="Sheet1", header=None)
print("Shape:", raw1.shape)
print(raw1.iloc[:4])


Shape: (12, 13)
       0                    1                2              3   \
0     NaN                  NaN              NaN            NaN   
1     NaN                 2016              NaN            NaN   
2  COUNTY  Harvested Area (HA)  Production (MT)  Yield (MT/HA)   
3   Bomet                32275            45517       1.410287   

                     4                5              6                    7   \
0                   NaN              NaN            NaN                  NaN   
1                  2017              NaN            NaN                 2018   
2   Harvested Area (HA)  Production (MT)  Yield (MT/HA)  Harvested Area (HA)   
3                 33792            56601       1.674982                33291   

                8             9                    10               11  \
0              NaN           NaN                  NaN              NaN   
1              NaN           NaN                 2020              NaN   
2  Production (MT)  Yield(MT/H

In [3]:
!pip install pandas numpy matplotlib scikit-learn qiskit qiskit-machine-learning qiskit-algorithms openpyxl xlrd


  Using cached pandas-3.0.5-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached matplotlib-3.11.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached scikit_learn-1.9.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached qiskit_machine_learning-0.9.1-py3-none-any.whl.metadata (13 kB)
  Using cached qiskit_algorithms-0.4.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached xlrd-2.0.2-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.64.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (123 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.2 k

In [4]:
import pandas as pd
print("pandas OK")

pandas OK


In [5]:
import pandas as pd

files_to_check = {
    "Yield": "data/Agr data/Annual Maize Yield Production 2012-2020.xlsx",
    "Bungoma Rainfall": "data/Agr data/BUNGOMA MEAN DAILY RAINFALL.xlsx 2016-2020.xls",
    "Nandi Rainfall": "data/Agr data/NANDI MEAN DAILY RAINFALL.xlsx 2016-2020.xlsx",
}

for label, path in files_to_check.items():
    print("=" * 70)
    print(label, "->", path)
    print("=" * 70)
    try:
        xls = pd.ExcelFile(path)
        print("Sheet names:", xls.sheet_names)
        for sheet in xls.sheet_names:
            df = pd.read_excel(path, sheet_name=sheet)
            print(f"\n--- Sheet: {sheet} ---")
            print("Shape:", df.shape)
            print("Columns:", df.columns.tolist())
            print(df.head(6))
    except Exception as e:
        print("ERROR reading file:", e)
    print("\n")

Yield -> data/Agr data/Annual Maize Yield Production 2012-2020.xlsx
Sheet names: ['Sheet1', 'Sheet4']

--- Sheet: Sheet1 ---
Shape: (11, 13)
Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12']
        Unnamed: 0           Unnamed: 1       Unnamed: 2     Unnamed: 3  \
0              NaN                 2016              NaN            NaN   
1           COUNTY  Harvested Area (HA)  Production (MT)  Yield (MT/HA)   
2            Bomet                32275            45517       1.410287   
3          Bungoma               100712           301068       2.989396   
4  Elgeyo/Marakwet                33315            75005       2.251388   
5         Kakamega                78344           172350       2.199913   

             Unnamed: 4       Unnamed: 5     Unnamed: 6           Unnamed: 7  \
0                  2017              NaN            NaN   

In [7]:
import pandas as pd

def load_rainfall(path):
    df = pd.read_excel(path, skiprows=1)  # skip the junk row, use real headers
    df.columns = ["date", "rainfall_mm"]
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["rainfall_mm"] = pd.to_numeric(df["rainfall_mm"], errors="coerce")
    return df

for label, path in [
    ("Bungoma", "data/Agr data/BUNGOMA MEAN DAILY RAINFALL.xlsx 2016-2020.xls"),
    ("Nandi", "data/Agr data/NANDI MEAN DAILY RAINFALL.xlsx 2016-2020.xlsx"),
]:
    df = load_rainfall(path)
    print(f"--- {label} ---")
    print("Total rows:", len(df))
    print("Date range:", df['date'].min(), "to", df['date'].max())
    print("Duplicate dates:", df['date'].duplicated().sum())
    print("Rows with missing date:", df['date'].isna().sum())
    print("Rows with missing rainfall:", df['rainfall_mm'].isna().sum())
    print("Sample:")
    print(df.head(3))
    print()

--- Bungoma ---
Total rows: 3774
Date range: 2016-03-01 00:00:00 to 2026-06-30 00:00:00
Duplicate dates: 0
Rows with missing date: 0
Rows with missing rainfall: 0
Sample:
        date  rainfall_mm
0 2016-03-01       0.0000
1 2016-03-02      11.6986
2 2016-03-03       0.2389

--- Nandi ---
Total rows: 1767
Date range: 2016-03-01 00:00:00 to 2020-12-31 00:00:00
Duplicate dates: 0
Rows with missing date: 0
Rows with missing rainfall: 0
Sample:
        date  rainfall_mm
0 2016-03-01       0.0000
1 2016-03-02      12.6001
2 2016-03-03       5.7750



In [6]:
for label, path in [
    ("Bungoma Rainfall", "data/Agr data/BUNGOMA MEAN DAILY RAINFALL.xlsx 2016-2020.xls"),
    ("Nandi Rainfall", "data/Agr data/NANDI MEAN DAILY RAINFALL.xlsx 2016-2020.xlsx"),
]:
    df = pd.read_excel(path)
    print(label)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.iloc[:3])
    print("-" * 50)

Bungoma Rainfall
Shape: (3775, 2)
Columns: ['Precipitation (CHIRPS)', 'Unnamed: 1']
  Precipitation (CHIRPS)          Unnamed: 1
0                   Date  Precipitation (mm)
1             2016-03-01                   0
2             2016-03-02             11.6986
--------------------------------------------------
Nandi Rainfall
Shape: (1768, 2)
Columns: ['Precipitation (CHIRPS)', 'Unnamed: 1']
  Precipitation (CHIRPS)         Unnamed: 1
0                   Date  precipitation(mm)
1             2016-03-01                  0
2             2016-03-02            12.6001
--------------------------------------------------


In [3]:
import pandas
import qiskit
import qiskit_machine_learning
print("All imports working!")

All imports working!


In [2]:
!pip install pandas numpy matplotlib scikit-learn qiskit qiskit-machine-learning qiskit-algorithms openpyxl xlrd

  Using cached pandas-3.0.5-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached matplotlib-3.11.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (80 kB)
  Using cached scikit_learn-1.9.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached qiskit_machine_learning-0.9.1-py3-none-any.whl.metadata (13 kB)
  Using cached qiskit_algorithms-0.4.0-py3-none-any.whl.metadata (4.7 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached xlrd-2.0.2-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached contourpy-1.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.64.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (123 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.2 k